# 📊 MeatGuard-S — XAI Faithfulness

**Phase 1 · RO5 (computational half)** — Quantify whether the Grad-CAM map identifies truly important pixels via **Insertion AUC** (higher = better) and **Deletion AUC** (lower = better). The **IoU vs. expert masks** is computed in Phase 3 once expert annotations exist.

Mirrors `src/xai/faithfulness.py`.

## 1. Setup & load model

In [ ]:
# Run once on a fresh Colab/Kaggle runtime (uncomment):
# !pip install torch torchvision timm einops scikit-learn matplotlib seaborn tqdm opencv-python shap scikit-image onnx

In [ ]:
import os, sys

# --- Path bootstrap: make ../src importable on local, Colab, and Kaggle ------
def add_src_to_path():
    here = os.getcwd()
    candidates = [
        os.path.abspath(os.path.join(here, "..", "src")),   # local: notebooks/ -> ../src
        os.path.abspath(os.path.join(here, "src")),         # run from repo root
        "/content/MeatGuard_II/src",                        # Colab (git clone target)
        "/kaggle/working/MeatGuard_II/src",                 # Kaggle
    ]
    for c in candidates:
        if os.path.isdir(c):
            if c not in sys.path:
                sys.path.insert(0, c)
            return c
    raise FileNotFoundError(
        "Could not locate the MeatGuard_II/src package. "
        "On Colab/Kaggle, clone the repo into /content (or /kaggle/working) first."
    )

SRC_DIR = add_src_to_path()
print("Using src package at:", SRC_DIR)

In [ ]:
import glob
import torch
import numpy as np
from PIL import Image

from model.architecture import MeatGuardS
from data.transforms import get_val_test_transform
from xai.gradcam import GradCAM
from xai.faithfulness import (
    calculate_insertion_auc, calculate_deletion_auc, calculate_iou,
)

In [ ]:
# Placeholder dataset (drop-in stand-in for the full MeatScan set).
# Override with:  os.environ["MEATGUARD_DATA"] = "/path/to/your/data"
DATA_DIR = os.environ.get("MEATGUARD_DATA", r"D:/Research_Work/MeatGuard/data/test")
ARTIFACTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "artifacts"))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("Data dir :", DATA_DIR)
print("Artifacts:", ARTIFACTS_DIR)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MeatGuardS(num_classes=2)
ckpt = os.path.join(ARTIFACTS_DIR, "meatguard_s_final.pth")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=device))
    print("Loaded weights:", ckpt)
else:
    print("No checkpoint found — using random weights (faithfulness will be meaningless).")
model.eval().to(device)

## 2. Sample image + Grad-CAM map

In [ ]:
candidates = sorted(glob.glob(os.path.join(DATA_DIR, "*", "*.jpg")) +
                    glob.glob(os.path.join(DATA_DIR, "*", "*.png")))
assert candidates, f"No images found under {DATA_DIR}"
img = Image.open(candidates[0]).convert("RGB")
input_tensor = get_val_test_transform()(img).unsqueeze(0).to(device)

target_class = int(torch.softmax(model(input_tensor), dim=1).argmax())
gcam = GradCAM(model, target_layer="stage4")
cam_map = gcam.generate(input_tensor, class_idx=target_class)
gcam.remove_hooks()

# Resize CAM to full image resolution so the masks index pixels 1:1
import cv2
cam_full = cv2.resize(cam_map, (input_tensor.shape[-1], input_tensor.shape[-2]))
print("target_class:", ['Fresh','Spoiled'][target_class], "| cam_full:", cam_full.shape)

## 3. Insertion / Deletion AUC

In [ ]:
ins_auc = calculate_insertion_auc(model, input_tensor, cam_full, target_class=target_class)
del_auc = calculate_deletion_auc(model, input_tensor, cam_full, target_class=target_class)
print(f"Insertion AUC : {ins_auc:.4f}   (higher is better)")
print(f"Deletion AUC  : {del_auc:.4f}   (lower is better)")

## 4. IoU vs. expert annotation (Phase 3 linkage)

In the field study, domain experts annotate spoilage regions. Load such a mask and compare it against the thresholded Grad-CAM map to validate the spatial explanation.

In [ ]:
EXPERT_MASK = os.path.join(ARTIFACTS_DIR, "expert_mask.npy")
if os.path.exists(EXPERT_MASK):
    expert_mask = np.load(EXPERT_MASK).astype(bool)
    iou = calculate_iou(cam_full > 0.5, expert_mask)
    print(f"Grad-CAM IoU vs expert: {iou:.4f}")
else:
    print("No expert_mask.npy yet — collected during the Phase 3 field study (RO5).")